    # Week 2: Calculus, probability and backprop by hand

    [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuranLearning/ai-llm-deep-dive/blob/main/notebooks/week-02-backprop-by-hand.ipynb)
    · [Lesson text](https://github.com/QuranLearning/ai-llm-deep-dive/blob/main/lessons/week-02.md) · [Learning Hub week page](https://claude.ai/artifact/VwTfCas4FMnB23QHuBUJhj#w-ai-llm-2)

    **By the end of this notebook you will:**
    - check derivatives numerically
- apply the chain rule to a small computational graph
- compute cross-entropy and perplexity
- verify that dL/dz = p − y for softmax + cross-entropy
- write the full backward pass of a 2-layer MLP and gradient-check it

    **How to use it:** work top to bottom. Each exercise has a starter cell (replace the `TODO`s) and a **Check** cell that tells you if you got it right.
    Stuck for more than 15 minutes? Ask the tutor on the week page, then peek at the **Solutions** section at the bottom.
    In Colab, use *File → Save a copy in Drive* so your work is saved.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
def ok(msg="Correct"): print("\u2705", msg)
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True)); return e / e.sum(axis=-1, keepdims=True)

## Exercise 1: numerical derivatives
Write `num_grad(f, x, eps=1e-5)` using the centred difference (f(x+ε) − f(x−ε)) / 2ε. Then write `dtanh(x)` analytically, using 1 − tanh(x)².

In [ ]:
def num_grad(f, x, eps=1e-5):
    # TODO
    raise NotImplementedError

def dtanh(x):
    # TODO
    raise NotImplementedError

In [ ]:
for x in [-2.0, -0.3, 0.0, 1.5]:
    assert np.isclose(num_grad(np.tanh, x), dtanh(x), atol=1e-8)
assert np.isclose(num_grad(lambda t: t**3, 2.0), 12.0, atol=1e-6)
ok("derivatives")

## Exercise 2: the chain rule on a tiny graph
For f(w, b) = tanh(w·x + b)², with x = 0.5, write the forward pass, then the backward pass node by node:
`a = w*x + b`, `h = tanh(a)`, `f = h**2`. Return `(f, df_dw, df_db)`.

In [ ]:
def tiny_graph(w, b, x=0.5):
    # forward
    # TODO: a, h, f
    # backward: start from df/df = 1 and apply the local rules
    # TODO: df_dh, df_da, df_dw, df_db
    raise NotImplementedError

In [ ]:
w, b = 1.3, -0.4
f, gw, gb = tiny_graph(w, b)
assert np.isclose(gw, num_grad(lambda t: tiny_graph(t, b)[0], w), atol=1e-7)
assert np.isclose(gb, num_grad(lambda t: tiny_graph(w, t)[0], b), atol=1e-7)
ok("chain rule")

## Exercise 3: cross-entropy and perplexity
`cross_entropy(P, y)`: P is (N, C) probabilities, y is (N,) integer targets; return the mean of −log P[i, y_i].
`perplexity(P, y)` = exp(cross-entropy). Sanity check from the lesson: a model that gives the right token 0.25 every time has perplexity 4.

In [ ]:
def cross_entropy(P, y):
    # TODO (hint: P[np.arange(len(y)), y])
    raise NotImplementedError

def perplexity(P, y):
    # TODO
    raise NotImplementedError

In [ ]:
P = np.full((10, 4), 0.25); y = rng.integers(0, 4, size=10)
assert np.isclose(cross_entropy(P, y), np.log(4)) and np.isclose(perplexity(P, y), 4)
assert np.isclose(cross_entropy(np.array([[0.9, 0.1]]), np.array([0])), 0.10536, atol=1e-4)
ok("cross-entropy and perplexity")

## Exercise 4: the gradient worth memorising
Write `ce_grad_logits(z, y)` for one example: logits z (C,), target index y. Return dL/dz = softmax(z) − onehot(y). The check compares it with a numerical gradient.

In [ ]:
def ce_grad_logits(z, y):
    # TODO
    raise NotImplementedError

In [ ]:
z, y = rng.normal(size=6), 2
L = lambda zz: -np.log(softmax(zz)[y])
g_num = np.array([num_grad(lambda t: L(np.r_[z[:i], t, z[i+1:]]), z[i]) for i in range(len(z))])
assert np.allclose(ce_grad_logits(z, y), g_num, atol=1e-7)
ok("p − y")

## Exercise 5: backprop for a 2-layer MLP (batch version)
Forward: `A1 = X W1 + b1`, `H = tanh(A1)`, `Z = H W2 + b2`, `P = softmax(Z)`, `L = mean CE`.
Write `mlp_loss_grads(params, X, y)` returning `(L, grads)` where grads has the same order and shapes as `params = [W1, b1, W2, b2]`. Remember to divide by the batch size N.

In [ ]:
def mlp_loss_grads(params, X, y):
    W1, b1, W2, b2 = params
    N = len(y)
    # TODO forward
    # TODO backward: dZ = (P - onehot) / N, then dW2, db2, dH, dA1, dW1, db1
    raise NotImplementedError

In [ ]:
d, h, C, N = 5, 16, 3, 32
params = [rng.normal(size=(d, h)) / np.sqrt(d), np.zeros(h), rng.normal(size=(h, C)) / np.sqrt(h), np.zeros(C)]
X, y = rng.normal(size=(N, d)), rng.integers(0, C, size=N)
L0, grads = mlp_loss_grads(params, X, y)
for k, (Pm, G) in enumerate(zip(params, grads)):
    assert G.shape == Pm.shape, f"grad {k} has shape {G.shape}, expected {Pm.shape}"
    for _ in range(10):
        idx = tuple(rng.integers(s) for s in Pm.shape); old = Pm[idx]
        Pm[idx] = old + 1e-5; Lp = mlp_loss_grads(params, X, y)[0]
        Pm[idx] = old - 1e-5; Lm = mlp_loss_grads(params, X, y)[0]
        Pm[idx] = old
        num = (Lp - Lm) / 2e-5
        assert abs(num - G[idx]) / max(1e-12, abs(num) + abs(G[idx])) < 1e-5, (k, idx, num, G[idx])
ok(f"MLP gradients match (loss {L0:.4f})")

## Reflect
In your weekly journal: which step of the backward pass did you get wrong first, and why? (That note is what the Thursday check-in reads.)

---
## Solutions

Try every exercise first. Run the cell below only to compare with your own version; it overwrites your functions, so re-run the Check cells afterwards.

In [ ]:
def num_grad(f, x, eps=1e-5):
    return (f(x + eps) - f(x - eps)) / (2 * eps)

def dtanh(x):
    return 1 - np.tanh(x) ** 2

def tiny_graph(w, b, x=0.5):
    a = w * x + b; h = np.tanh(a); f = h ** 2
    df_dh = 2 * h
    df_da = df_dh * (1 - h ** 2)
    return f, df_da * x, df_da * 1.0

def cross_entropy(P, y):
    return float(np.mean(-np.log(P[np.arange(len(y)), y])))

def perplexity(P, y):
    return float(np.exp(cross_entropy(P, y)))

def ce_grad_logits(z, y):
    g = softmax(z).copy(); g[y] -= 1; return g

def mlp_loss_grads(params, X, y):
    W1, b1, W2, b2 = params
    N = len(y)
    H = np.tanh(X @ W1 + b1)
    P = softmax(H @ W2 + b2)
    L = float(np.mean(-np.log(P[np.arange(N), y])))
    dZ = P.copy(); dZ[np.arange(N), y] -= 1; dZ /= N
    dW2, db2 = H.T @ dZ, dZ.sum(0)
    dA1 = (dZ @ W2.T) * (1 - H ** 2)
    dW1, db1 = X.T @ dA1, dA1.sum(0)
    return L, [dW1, db1, dW2, db2]